# Chapter 10 — Framework Interoperability (v2026)

> **LangChain 1.x / 2026 refresh.** Dual-mode secrets, optional LangSmith tracing, pinned core deps where applicable, and a standardized footer (Limitations & safety + cleanup + exercises). **REFACTORED_FRAMEWORK_INTEROP_V2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/CHDIR/FNAME)

## Learning objectives
- Implement one task with a standard interface
- Show a LangChain/LangGraph adapter and conceptual adapters for alternatives
- Compare frameworks by task and operations needs

> **Runtime / cost / data.** Offline; uses langchain/langgraph only, with light conceptual adapters for alternatives.

## Environment setup

In [ ]:
import os

# Secrets are read from Colab Secrets if available, else from a local .env
try:
    from google.colab import userdata  # type: ignore

    def get_secret(name, default=""):
        return userdata.get(name) or default
except Exception:
    try:
        from dotenv import load_dotenv  # type: ignore

        load_dotenv()
    except Exception:
        pass

    def get_secret(name, default=""):
        return os.environ.get(name, default)

OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Optional LangSmith tracing (set LANGCHAIN_API_KEY to enable)
LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter10-framework-interop"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_PROJECT", LANGSMITH_PROJECT)
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")

## A standard task interface

Define a single, framework-agnostic contract — `retrieve_and_answer(question) -> {answer, sources}` — then implement it per framework. This makes swap-outs and comparisons objective.

In [ ]:
from typing import Protocol, List, Dict

class RAGBackend(Protocol):
    name: str
    def retrieve_and_answer(self, question: str) -> Dict:
        ...

CORPUS = [
    {"id": "d1", "text": "Metformin is first-line for type 2 diabetes."},
    {"id": "d2", "text": "SGLT2 inhibitors reduce cardiovascular events."},
    {"id": "d3", "text": "GLP-1 agonists support weight reduction."},
]

def keyword_retrieve(q, docs, k=2):
    toks = set(q.lower().split())
    return sorted(docs, key=lambda d: -len(toks & set(d['text'].lower().split())))[:k]

In [ ]:
# LangChain-style adapter (uses real langchain_core for the prompt)
from langchain_core.prompts import ChatPromptTemplate

class LangChainBackend:
    name = "langchain"
    def retrieve_and_answer(self, question):
        hits = keyword_retrieve(question, CORPUS)
        ctx = "\n".join(h["text"] for h in hits)
        prompt = ChatPromptTemplate.from_template("Answer using context: {ctx}\nQ: {q}")
        _ = prompt.format(ctx=ctx, q=question)  # prompt is rendered; answer is mocked offline
        return {"answer": hits[0]["text"], "sources": [h["id"] for h in hits]}

lc = LangChainBackend()
print(lc.name, lc.retrieve_and_answer("Which drugs help cardiovascular outcomes?"))

In [ ]:
# Conceptual adapters for alternatives (light mocks; no heavy installs)
class LlamaIndexBackend:
    name = "llamaindex"
    def retrieve_and_answer(self, question):
        hits = keyword_retrieve(question, CORPUS)  # stand-in for VectorStoreIndex
        return {"answer": hits[0]["text"], "sources": [h["id"] for h in hits]}

class HaystackBackend:
    name = "haystack"
    def retrieve_and_answer(self, question):
        hits = keyword_retrieve(question, CORPUS)  # stand-in for a Pipeline
        return {"answer": hits[0]["text"], "sources": [h["id"] for h in hits]}

backends = [LangChainBackend(), LlamaIndexBackend(), HaystackBackend()]
q = "Which class supports weight loss?"
for b in backends:
    print(b.name, "->", b.retrieve_and_answer(q))

## Comparing frameworks by task & operations

| Axis | LangChain/LangGraph | LlamaIndex | Haystack |
|------|--------------------|-----------| ----------|
| Best for | agents, stateful graphs | document RAG/indexing | production pipelines |
| State mgmt | checkpointers | limited | pipeline-centric |
| Observability | LangSmith | LlamaTrace | OpenTelemetry |
| Ops | flexible, DIY | simple RAG | strong serving |

Choose by **task requirements and ops needs** (state, observability, deployment), not popularity.

In [ ]:
# Adapter conformance check: all backends satisfy the same contract
def check_contract(backend):
    out = backend.retrieve_and_answer("test")
    assert set(out) >= {"answer", "sources"}, backend.name
    return backend.name + " OK"

for b in backends:
    print(check_contract(b))

## Limitations & safety

- **Enterprise / research-support only.** Human review is required before any production, clinical, or compliance decision.
- **Synthetic / de-identified data only.** Real PHI/PII requires governance and access controls.

In [ ]:
# Cleanup: drop references and free memory.
import gc

for _name in ["llm", "chain", "model", "agent", "app", "manifest", "report"]:
    globals().pop(_name, None)

gc.collect()
print("Cleanup complete.")

## Exercises

<details><summary>Q1. Why is a run manifest essential for reproducibility in an enterprise pipeline?</summary>
It captures corpus/index versions, prompts, and model/tool versions, so a past result can be audited, diffed, or re-run deterministically when models or data change.
</details>

<details><summary>Q2. Why treat guardrail / injection detections as drafts rather than automatic enforcement?</summary>
Detectors have false positives and negatives. In regulated settings, an error can block legitimate work or leak PHI, so a human confirms before enforcement.
</details>

<details><summary>Q3. Why compare frameworks by task and operations needs instead of popularity?</summary>
The best fit depends on state management, observability, deployment, and team skill — not download counts. A task-driven matrix makes the trade-offs explicit and defensible.
</details>

### Task A — Add a `CrewAIBackend` mock that returns a 'crew' multi-step answer, keeping the same contract.

### Task B — Time each backend over the same query set and report mean latency in a table.

### Task C — Add a `health_check()` method to the interface and implement it for all backends.

### Task D — Write a config-driven factory that instantiates a backend by name from a dict.